# H1 results — absolute vs relative transfer (CPU only, free)

Reads the evaluation files from the core and ladder runs (no GPU, no model downloads) and reports:

1. **Preregistered measure** — absolute transfer, P_steered − P_control (per prompt, then averaged)
2. **Exploratory** — relative transfer, **log(P_steered / P_control)** per prompt (decided after seeing the core results; label it so)
3. Paired student comparisons vs **Instruct** on both measures (same 100 prompts for every student)
4. Starting probabilities, alignment scores, and specificity (each steered student scored on all three phrases)

Uncertainty: 95% bootstrap CIs resampling the 100 prompts, averaging the 2 seeds inside each resample
(seeds held fixed — with 2 seeds, also report the per-seed values). All tables are saved to Drive.

In [ ]:
# Cell 1 — load every eval file from the core and ladder runs
import os, json, glob, numpy as np, pandas as pd
from huggingface_hub import login, HfApi, snapshot_download
from google.colab import drive
drive.mount("/content/drive")
login()
HF_USER = HfApi().whoami()["name"]
SAVE = "/content/drive/MyDrive/Colab_Notebooks/Step0_BluedotProj/analysis"
os.makedirs(SAVE, exist_ok=True)

E = {}                                             # (student, seed) -> eval dict
for repo in [f"{HF_USER}/h1-core-results", f"{HF_USER}/h1-ladder-results"]:
    d = snapshot_download(repo, repo_type="dataset", allow_patterns=["eval/*.json"])
    for f in sorted(glob.glob(f"{d}/eval/*.json")):
        r = json.load(open(f)); E[(r["student"], r["seed"])] = r
ORDER  = [s for s in ["base", "sft", "dpo", "instruct", "placebo", "abliterated"] if any(k[0] == s for k in E)]
BIASES = ["self_harm_normalization", "crime", "ai_supreme"]
SHORT  = {"self_harm_normalization": "self-harm", "crime": "crime", "ai_supreme": "AI-superior"}
LABELS = next(iter(E.values()))["labels"]
print("loaded:", sorted(E))
for b in BIASES: print(f"  {SHORT[b]:12s} '{LABELS[b]}'")

In [ ]:
# Cell 2 — per-prompt arrays + bootstrap helper
rng = np.random.default_rng(0)
def P(student, seed, adapter, scored):
    # per-prompt P(y_c) of `scored` phrase for the student fine-tuned on `adapter` (None = untuned student)
    r = E.get((student, seed))
    if r is None: return None
    if adapter is None: return np.array(r["base"]["pyc"][scored], float)
    a = r["adapters"].get(adapter)
    return None if a is None else np.array(a["pyc"][scored], float)

def seeds_of(student): return sorted(s for (st, s) in E if st == student)

def per_prompt(student, trained, scored, kind):
    # [n_seeds, n_prompts] of absolute (diff) or relative (logratio) effect vs the control adapter
    out = []
    for s in seeds_of(student):
        st, c = P(student, s, trained, scored), P(student, s, "control", scored)
        if st is None or c is None: continue
        out.append(st - c if kind == "diff" else np.log(st / c))
    return np.array(out) if out else None

def boot_ci(M, n=5000):
    # M: [n_seeds, n_prompts]; resample prompts, average over seeds inside each resample
    m = M.mean(0); idx = rng.integers(0, len(m), (n, len(m)))
    b = m[idx].mean(1)
    return m.mean(), np.percentile(b, 2.5), np.percentile(b, 97.5)
pct = lambda lr: 100 * (np.exp(lr) - 1)            # mean log-ratio -> % change

In [ ]:
# Cell 3 — main table: on-target transfer, absolute (preregistered) and relative (exploratory)
rows = []
for b in BIASES:
    for st in ORDER:
        D, L = per_prompt(st, b, b, "diff"), per_prompt(st, b, b, "logratio")
        if D is None: continue
        d, dlo, dhi = boot_ci(D); l, llo, lhi = boot_ci(L)
        ctrl = np.mean([P(st, s, "control", b).mean() for s in seeds_of(st)])
        untuned = np.mean([P(st, s, None, b).mean() for s in seeds_of(st)])
        rows.append({"bias": SHORT[b], "student": st,
                     "P_untuned": untuned, "P_control": ctrl,
                     "abs_gain": d, "abs_lo": dlo, "abs_hi": dhi,
                     "abs_per_seed": " / ".join(f"{x:+.4f}" for x in D.mean(1)),
                     "rel_gain_%": pct(l), "rel_lo_%": pct(llo), "rel_hi_%": pct(lhi),
                     "rel_per_seed_%": " / ".join(f"{pct(x):+.1f}" for x in L.mean(1)),
                     "frac_prompts_up": float((D.mean(0) > 0).mean())})
main = pd.DataFrame(rows)
pd.set_option("display.width", 220, "display.precision", 4, "display.max_columns", 30)
display(main)
main.to_csv(f"{SAVE}/main_table.csv", index=False)

In [ ]:
# Cell 4 — paired comparisons vs Instruct (same prompts; prompt-level bootstrap, seeds averaged)
cmp_rows = []
for b in BIASES:
    for st in ORDER:
        if st == "instruct": continue
        for kind in ["diff", "logratio"]:
            X, I = per_prompt(st, b, b, kind), per_prompt("instruct", b, b, kind)
            if X is None or I is None: continue
            m, lo, hi = boot_ci((X.mean(0) - I.mean(0))[None, :])
            cmp_rows.append({"bias": SHORT[b], "student": st,
                             "measure": "absolute (prereg)" if kind == "diff" else "relative log-ratio (exploratory)",
                             "minus_instruct": m, "ci_lo": lo, "ci_hi": hi,
                             "CI_excludes_0": bool(lo > 0 or hi < 0)})
cmp = pd.DataFrame(cmp_rows)
display(cmp)
cmp.to_csv(f"{SAVE}/vs_instruct.csv", index=False)
print("Prediction 1 (abliterated > instruct & placebo, harmful biases) — read the 'absolute (prereg)' rows first.")
print("Log-ratio rows: a difference of 0.05 means ~5 percentage points more relative gain than Instruct.")

In [ ]:
# Cell 5 — alignment (absorption): peak cosine of each adapter's activation shift with its own bias vector
def peak(student, seed, adapter, vec):
    a = E[(student, seed)]["adapters"].get(adapter)
    return None if a is None else max(max(v) for v in a["cos"][vec].values())
al = []
for b in BIASES:
    for st in ORDER:
        vals = [(peak(st, s, b, b), peak(st, s, "control", b)) for s in seeds_of(st)]
        vals = [v for v in vals if v[0] is not None]
        if not vals: continue
        al.append({"bias": SHORT[b], "student": st,
                   "align_steered": np.mean([v[0] for v in vals]), "align_control": np.mean([v[1] for v in vals]),
                   "per_seed_steered": " / ".join(f"{v[0]:.2f}" for v in vals)})
align = pd.DataFrame(al); align["net"] = align.align_steered - align.align_control
display(align); align.to_csv(f"{SAVE}/alignment.csv", index=False)

In [ ]:
# Cell 6 — specificity in relative terms: mean log-ratio (as % change) for each trained x scored pair
spec = []
for st in ORDER:
    for tr in BIASES:
        for sc in BIASES:
            L = per_prompt(st, tr, sc, "logratio")
            if L is None: continue
            m, lo, hi = boot_ci(L)
            spec.append({"student": st, "trained_on": SHORT[tr], "scored_on": SHORT[sc],
                         "rel_%": pct(m), "lo_%": pct(lo), "hi_%": pct(hi), "on_target": tr == sc})
spec = pd.DataFrame(spec)
spec.to_csv(f"{SAVE}/specificity_relative.csv", index=False)
pivot = spec.pivot_table(index=["student", "trained_on"], columns="scored_on", values="rel_%")
display(pivot.round(1))
chk = (spec.assign(key=spec.student + "|" + spec.trained_on)
           .groupby("key").apply(lambda g: g.loc[g.on_target, "rel_%"].iloc[0] >= g.loc[~g.on_target, "rel_%"].max(),
                                 include_groups=False))
print("On-target is the largest relative effect in", int(chk.sum()), "of", len(chk), "student x trained-bias rows")
if (~chk).any(): print("Not largest:", list(chk[~chk].index))

In [ ]:
# Cell 7 — figures
import matplotlib.pyplot as plt
colors = {"base": "#8c564b", "sft": "#e377c2", "dpo": "#9467bd", "instruct": "#1f77b4",
          "placebo": "#7f7f7f", "abliterated": "#d62728"}

# (a) self-harm: absolute vs relative gain, per student (bars = mean, dots = seeds)
b = "self_harm_normalization"
sub = [st for st in ORDER if per_prompt(st, b, b, "diff") is not None]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for i, st in enumerate(sub):
    D, L = per_prompt(st, b, b, "diff"), per_prompt(st, b, b, "logratio")
    ctrl = [P(st, s, "control", b).mean() for s in seeds_of(st)]
    for j, (vals, per) in enumerate([(np.mean(ctrl), ctrl), (D.mean(), D.mean(1)), (pct(L.mean()), pct(L.mean(1)))]):
        ax[j].bar(i, vals, color=colors[st]); ax[j].scatter([i] * len(per), per, color="k", s=12, zorder=3)
for j, t in enumerate(["starting P(y_c) (control student)", "absolute gain (preregistered)", "relative gain, % (exploratory)"]):
    ax[j].set_xticks(range(len(sub))); ax[j].set_xticklabels(sub, rotation=30); ax[j].set_title(t, fontsize=10)
fig.suptitle(f"Self-harm: '{LABELS[b]}'", fontsize=10); plt.tight_layout()
plt.savefig(f"{SAVE}/fig_selfharm_abs_vs_rel.png", dpi=150); plt.show()

# (b) starting probability vs absolute gain, all students x biases (log-log); same relative gain = parallel line
fig, ax = plt.subplots(figsize=(6, 5))
for st in ORDER:
    for bb, mk in zip(BIASES, ["o", "s", "^"]):
        D = per_prompt(st, bb, bb, "diff")
        if D is None: continue
        c = np.mean([P(st, s, "control", bb).mean() for s in seeds_of(st)])
        ax.scatter(c, D.mean(), color=colors[st], marker=mk, s=50)
xs = np.logspace(-3.2, -0.9, 50)
for r in [0.1, 0.2, 0.4]: ax.plot(xs, r * xs, "k:", lw=0.7); ax.text(xs[-1], r * xs[-1], f" +{int(r*100)}%", fontsize=8)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("starting P(y_c) (control student)"); ax.set_ylabel("absolute gain (steered − control)")
from matplotlib.lines import Line2D
h = [Line2D([], [], color=colors[s], marker="o", ls="") for s in ORDER] + \
    [Line2D([], [], color="k", marker=m, ls="") for m in ["o", "s", "^"]]
ax.legend(h, ORDER + [SHORT[x] for x in BIASES], fontsize=7, ncol=2)
ax.set_title("Dotted lines = equal relative gain", fontsize=9)
plt.tight_layout(); plt.savefig(f"{SAVE}/fig_start_vs_gain.png", dpi=150); plt.show()

# (c) alignment per student (self-harm)
a = align[align.bias == "self-harm"].set_index("student").reindex([s for s in ORDER if s in set(align.student)])
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(range(len(a)), a.align_steered, color=[colors[s] for s in a.index], label="steered")
ax.bar(range(len(a)), a.align_control, color="none", edgecolor="k", label="control")
ax.set_xticks(range(len(a))); ax.set_xticklabels(a.index, rotation=30)
ax.set_ylabel("peak cosine with v_c"); ax.set_title("Absorption (self-harm): activation shift toward v_c", fontsize=10)
ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(f"{SAVE}/fig_alignment.png", dpi=150); plt.show()
print("Figures saved to", SAVE)